In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from pyspark.sql import functions as F

# Load Bronze, Silver and Gold datasets
df_bronze = spark.table("bronze_ed_quarterly")
df_silver = spark.table("silver_ed_quarterly")
df_gold = spark.table("gold_ed_hospital_quarterly")
df_triage = spark.table("gold_ed_triage_performance")

# Display record counts
print("Bronze records:", df_bronze.count())
print("Silver records:", df_silver.count())
print("Gold hospital records:", df_gold.count())
print("Gold triage records:", df_triage.count())


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 3, Finished, Available, Finished, False)

Bronze records: 6930
Silver records: 6930
Gold hospital records: 1155
Gold triage records: 5775


In [6]:

from pyspark.sql import functions as F

# Define business keys for duplicate detection
duplicate_checks = {
    "Bronze": (
        df_bronze,
        ["facility_code", "reporting_quarter", "triage_category"]
    ),
    "Silver": (
        df_silver,
        ["facility_code", "quarter_end_month", "triage_category"]
    ),
    "Gold Hospital": (
        df_gold,
        ["facility_code", "quarter_end_month"]
    ),
    "Gold Triage": (
        df_triage,
        ["facility_code", "quarter_end_month", "triage_category"]
    )
}

results = []

for table_name, (df, keys) in duplicate_checks.items():

    missing_keys = [k for k in keys if k not in df.columns]

    if missing_keys:
        results.append({
            "table": table_name,
            "status": "SCHEMA_ERROR",
            "duplicate_groups": None,
            "missing_columns": ", ".join(missing_keys)
        })
        continue

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_count = duplicates.count()

    results.append({
        "table": table_name,
        "status": "PASS" if duplicate_count == 0 else "FAIL",
        "duplicate_groups": duplicate_count,
        "missing_columns": ""
    })

display(spark.createDataFrame(results))


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc08ab5f-f883-4692-8a68-f21b7aa113d6)

In [3]:

# Check Bronze column names
print(df_bronze.columns)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 5, Finished, Available, Finished, False)

['facility_code', 'facility_name', 'reporting_quarter', 'triage_category', 'number_of_attendances', 'attendance_variation_pct', 'seen_within_target_pct', 'median_wait_minutes', 'did_not_wait_pct', 'admitted_from_ed_pct', 'admissions_within_4h_pct', 'ed_stay_within_4h_pct', 'left_after_treatment_count', 'left_after_treatment_pct', 'did_not_wait_count', 'median_wait_traffic_light', 'source_file']


In [4]:

from pyspark.sql import functions as F

# Check duplicate records in Bronze
bronze_duplicates = (
    df_bronze
    .groupBy(
        "facility_code",
        "reporting_quarter",
        "triage_category"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = bronze_duplicates.count()

print("Bronze duplicate groups:", duplicate_count)
print(
    "Status:",
    "PASS" if duplicate_count == 0 else "FAIL"
)

if duplicate_count > 0:
    display(bronze_duplicates)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 6, Finished, Available, Finished, False)

Bronze duplicate groups: 0
Status: PASS


In [7]:

from pyspark.sql import functions as F

# Clinical indicators in the Silver dataset
clinical_columns = [
    "number_of_attendances",
    "attendance_variation_pct",
    "seen_within_target_pct",
    "median_wait_minutes",
    "did_not_wait_pct",
    "admitted_from_ed_pct",
    "admissions_within_4h_pct",
    "ed_stay_within_4h_pct",
    "left_after_treatment_pct"
]

total_records = df_silver.count()

missing_results = []

for column in clinical_columns:

    if column not in df_silver.columns:
        print(f"Column not found: {column}")
        continue

    missing_count = df_silver.filter(
        F.col(column).isNull()
    ).count()

    missing_pct = (
        missing_count / total_records * 100
        if total_records > 0 else 0
    )

    missing_results.append((
        column,
        int(missing_count),
        round(missing_pct, 2)
    ))

# Explicit schema for stable PySpark output
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType
)

schema = StructType([
    StructField("column_name", StringType(), False),
    StructField("missing_count", LongType(), False),
    StructField("missing_percentage", DoubleType(), False)
])

df_missing = spark.createDataFrame(
    missing_results,
    schema=schema
)

display(
    df_missing.orderBy(
        F.desc("missing_count")
    )
)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c803a8e-2eec-4cea-95cc-352823efd9a0)

In [8]:

from pyspark.sql import functions as F

# Percentage indicators must be between 0 and 100
percentage_columns = [
    "seen_within_target_pct",
    "did_not_wait_pct",
    "admitted_from_ed_pct",
    "admissions_within_4h_pct",
    "ed_stay_within_4h_pct",
    "left_after_treatment_pct"
]

results = []

# Validate percentage ranges
for column in percentage_columns:
    if column not in df_silver.columns:
        results.append((column, "SCHEMA_ERROR", None))
        continue

    invalid_count = df_silver.filter(
        F.col(column).isNotNull()
        & (
            (F.col(column) < 0)
            | (F.col(column) > 100)
            | F.isnan(F.col(column).cast("double"))
        )
    ).count()

    results.append((
        column,
        "PASS" if invalid_count == 0 else "FAIL",
        invalid_count
    ))

# Validate nonnegative attendance and waiting-time values
for column in ["number_of_attendances", "median_wait_minutes"]:
    if column not in df_silver.columns:
        results.append((column, "SCHEMA_ERROR", None))
        continue

    invalid_count = df_silver.filter(
        F.col(column).isNotNull()
        & (
            (F.col(column) < 0)
            | F.isnan(F.col(column).cast("double"))
        )
    ).count()

    results.append((
        column,
        "PASS" if invalid_count == 0 else "FAIL",
        invalid_count
    ))

from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType
)

schema = StructType([
    StructField("clinical_indicator", StringType(), False),
    StructField("status", StringType(), False),
    StructField("invalid_records", LongType(), True)
])

df_range_checks = spark.createDataFrame(results, schema)

display(df_range_checks)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b6ddfad6-9e64-48ed-b2b0-92b12751f0f7)

In [9]:

from pyspark.sql import functions as F

# 1. Sum attendances across ATS categories
df_triage_totals = (
    df_triage
    .groupBy("facility_code", "quarter_end_month")
    .agg(
        F.sum("attendances").alias("triage_total")
    )
)

# 2. Select reported hospital-level attendances
df_hospital_totals = (
    df_gold
    .select(
        "facility_code",
        "quarter_end_month",
        F.col("total_attendances").alias("hospital_total")
    )
)

# 3. Join both datasets, retaining unmatched keys
df_reconciliation = (
    df_hospital_totals
    .join(
        df_triage_totals,
        on=["facility_code", "quarter_end_month"],
        how="full_outer"
    )
    .withColumn(
        "difference",
        F.col("hospital_total") - F.col("triage_total")
    )
    .withColumn(
        "status",
        F.when(
            F.col("hospital_total").isNull()
            | F.col("triage_total").isNull(),
            "MISSING_RECORD"
        )
        .when(F.col("difference") == 0, "PASS")
        .otherwise("FAIL")
    )
)

# 4. Summarise reconciliation
print("Total facility-quarter groups:", df_reconciliation.count())

display(
    df_reconciliation
    .groupBy("status")
    .count()
)

# 5. Show discrepancies if any
display(
    df_reconciliation
    .filter(F.col("status") != "PASS")
    .limit(20)
)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 11, Finished, Available, Finished, False)

Total facility-quarter groups: 1155


SynapseWidget(Synapse.DataFrame, 0fff12d8-fbcd-456f-b8e0-a9dac37b9411)

SynapseWidget(Synapse.DataFrame, 8f3bc3f7-93ac-4121-b861-5387aeb74941)

In [11]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType
)

quality_results = []

# 1. Duplicate detection
duplicate_checks = {
    "Bronze": (
        df_bronze,
        ["facility_code", "reporting_quarter", "triage_category"]
    ),
    "Silver": (
        df_silver,
        ["facility_code", "quarter_end_month", "triage_category"]
    ),
    "Gold Hospital": (
        df_gold,
        ["facility_code", "quarter_end_month"]
    ),
    "Gold Triage": (
        df_triage,
        ["facility_code", "quarter_end_month", "triage_category"]
    )
}

for table_name, (df, keys) in duplicate_checks.items():

    duplicate_count = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    quality_results.append((
        "Duplicate Detection",
        table_name,
        "PASS" if duplicate_count == 0 else "FAIL",
        int(duplicate_count),
        None
    ))

# 2. Missing-value monitoring
for row in df_missing.collect():
    quality_results.append((
        "Missing Values",
        row["column_name"],
        "WARNING" if row["missing_count"] > 0 else "PASS",
        int(row["missing_count"]),
        float(row["missing_percentage"])
    ))

# 3. Clinical KPI range validation
for row in df_range_checks.collect():
    quality_results.append((
        "KPI Range Validation",
        row["clinical_indicator"],
        row["status"],
        int(row["invalid_records"])
        if row["invalid_records"] is not None else None,
        None
    ))

# 4. Attendance reconciliation
reconciliation_summary = (
    df_reconciliation
    .groupBy("status")
    .count()
    .collect()
)

for row in reconciliation_summary:
    quality_results.append((
        "Attendance Reconciliation",
        "Hospital vs ATS totals",
        row["status"],
        int(row["count"]),
        None
    ))

# Explicit schema
schema = StructType([
    StructField("check_type", StringType(), False),
    StructField("indicator", StringType(), False),
    StructField("status", StringType(), False),
    StructField("record_count", LongType(), True),
    StructField("percentage", DoubleType(), True)
])

df_quality_summary = (
    spark.createDataFrame(quality_results, schema)
    .withColumn("checked_at", F.current_timestamp())
)

display(df_quality_summary)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7cd9bc6-db29-48ce-ae48-fe00a51689ed)

In [12]:

# Save consolidated data quality results

(
    df_quality_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_ed_data_quality_summary")
)

print("Data quality summary saved successfully!")

# Verify saved results
df_saved_quality = spark.table(
    "gold_ed_data_quality_summary"
)

print("Total quality checks:", df_saved_quality.count())

display(
    df_saved_quality.groupBy("check_type", "status")
    .count()
    .orderBy("check_type", "status")
)


StatementMeta(, b503d8d2-d8c6-4a95-ac06-5865d620fae5, 14, Finished, Available, Finished, False)

Data quality summary saved successfully!
Total quality checks: 22


SynapseWidget(Synapse.DataFrame, 63956705-9980-410c-bda1-113945de7cd4)